# Native Qwen vs NOSE LoRA + Head: odor-vector algebra
1. **Native Qwen3 (4096-D):** unmodified model.
2. **NOSE LoRA + Head (512-D):** trained LoRA and contrastive descriptor head.

Candidates beginning with the first letter of either operand are excluded, matching the original analysis. Rank and rank percentile are lower-is-better.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress")
import logging
logging.getLogger("Uni-Mol Tools").setLevel(logging.ERROR)
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from pathlib import Path
import json

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

import nose as nose_package
from nose import NOSEPipeline

ROOT = Path(nose_package.__file__).resolve().parents[1]
descriptors = (ROOT / "demos/assets/odor_descriptors.txt").read_text().splitlines()
tests = json.loads((ROOT / "demos/assets/odor_algebra_tests.json").read_text())
assert len(descriptors) == 1086
index = {word: position for position, word in enumerate(descriptors)}
MODEL_COLORS = {
    "Native Qwen3": "#7f8c8d",
    "NOSE LoRA + Head": "#2980b9",
}

print(f"Fixed candidate vocabulary: {len(descriptors)} descriptors")
print(f"Registered tests: {len(tests)} queries, {sum(len(v['answers']) for v in tests.values())} answers")

Fixed candidate vocabulary: 1086 descriptors
Registered tests: 6 queries, 23 answers


In [2]:
pipeline = NOSEPipeline.from_pretrained()
representations = {
    "Native Qwen3": pipeline.encode_native(
        descriptors,
        batch_size=32,
        normalize=False,
    ).numpy(),
    "NOSE LoRA + Head": pipeline.encode_descriptors(
        descriptors,
        batch_size=32,
        normalize=False,
    ).numpy(),
}

def evaluate_model(model_name, embeddings):
    rows = []
    normalized = embeddings / np.linalg.norm(
        embeddings, axis=1, keepdims=True
    ).clip(1e-12)
    for query, spec in tests.items():
        left, right = (embeddings[index[word]] for word in spec["operands"])
        vector = left + right if spec["operation"] == "add" else left - right
        vector = vector / max(float(np.linalg.norm(vector)), 1e-12)
        excluded_initials = {word[0].lower() for word in spec["operands"]}
        candidate_indices = np.array([
            i for i, descriptor in enumerate(descriptors)
            if descriptor and descriptor[0].lower() not in excluded_initials
        ])
        scores = normalized[candidate_indices] @ vector
        ranked_indices = candidate_indices[np.argsort(-scores)]
        ranks = {int(candidate): rank for rank, candidate in enumerate(ranked_indices, 1)}
        for answer in spec["answers"]:
            answer_index = index[answer]
            rank = ranks[answer_index]
            rows.append({
                "model": model_name,
                "query": query,
                "answer": answer,
                "expression": f"{query} = {answer}",
                "rank": rank,
                "candidate_count": len(candidate_indices),
                "rank_percentile": rank / len(candidate_indices) * 100,
            })
    return rows

results = pd.DataFrame([
    row
    for model_name, embeddings in representations.items()
    for row in evaluate_model(model_name, embeddings)
])

# Grouped result table with visual separation between queries.
from IPython.display import HTML

def render_grouped_table(results_df, model_names):
    header = (
        "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
        "<caption style='font-weight:bold;font-size:14px;padding:8px'>"
        "Registered odor-algebra answers — rank percentile (lower is better)</caption>"
        "<thead><tr style='border-bottom:2px solid #333'>"
        "<th style='text-align:left;padding:6px 10px'>Query</th>"
        "<th style='text-align:left;padding:6px 10px'>Answer</th>"
    )
    for m in model_names:
        header += f"<th style='text-align:right;padding:6px 10px'>{m}</th>"
    header += "</tr></thead><tbody>"

    rows_html = ""
    queries = list(dict.fromkeys(results_df["query"]))
    bg_colors = ["#ffffff", "#f8f9fa"]
    for qi, query in enumerate(queries):
        bg = bg_colors[qi % 2]
        subset = results_df[results_df["query"] == query]
        answers = list(dict.fromkeys(subset["answer"]))
        for ai, answer in enumerate(answers):
            row_data = subset[subset["answer"] == answer]
            border_top = "border-top:2px solid #dee2e6;" if ai == 0 else ""
            q_cell = f"<td rowspan='{len(answers)}' style='padding:6px 10px;font-weight:600;vertical-align:top;{border_top}background:{bg}'>{query}</td>" if ai == 0 else ""
            rows_html += f"<tr style='background:{bg};{border_top}'>"
            rows_html += q_cell
            rows_html += f"<td style='padding:4px 10px'>{answer}</td>"
            for m in model_names:
                val = row_data[row_data["model"] == m]
                if len(val):
                    r = val.iloc[0]
                    pct = r["rank_percentile"]
                    rank = int(r["rank"])
                    color = "#16a34a" if pct < 5 else ("#d97706" if pct < 20 else "#6b7280")
                    rows_html += f"<td style='text-align:right;padding:4px 10px;color:{color}'>{pct:.3f}%&nbsp;&nbsp;<small style='color:#9ca3af'>(#{rank})</small></td>"
                else:
                    rows_html += "<td></td>"
            rows_html += "</tr>"
        # Query subtotal row
        rows_html += f"<tr style='background:{bg};border-bottom:1px solid #e5e7eb'><td></td><td style='padding:4px 10px;font-style:italic;color:#6b7280'>mean</td>"
        for m in model_names:
            q_results = results_df[(results_df["query"] == query) & (results_df["model"] == m)]
            mean_pct = q_results["rank_percentile"].mean()
            rows_html += f"<td style='text-align:right;padding:4px 10px;font-style:italic;color:#6b7280'>{mean_pct:.1f}%</td>"
        rows_html += "</tr>"

    rows_html += "</tbody></table>"
    return header + rows_html

display(HTML(render_grouped_table(results, list(representations))))


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 60.28it/s]
